# Meeting 2: A single-pass poker decision model

Our canonical synthesis of four Meeting 2 notebooks. We jointly encode
`{state, question, decisions}`, mean-pool each candidate's contextual tokens,
and apply **one shared scoring head**. Cross-entropy trains the encoder and head;
softmax displays probabilities. No autoregressive answer generation.

**Goal:** imitate PokerBench's GTO-derived reference actions on held-out situations.
Single-action labels do not recover full solver mixing probabilities or establish
unexploitable play. Prior-hand opponent profiles and tilt are future work; the current
hand's betting sequence is already part of the state.

**Local setup**, from the repository root:
```sh
uv sync --locked --extra analysis --extra ml
uv run --locked --extra analysis --extra ml jupyter lab notebooks/02_pokerbench_decision_model.ipynb
```
Select the project environment, restart the kernel, and Run All. On Colab select a GPU
runtime first. The next cell installs this checkout only in Colab. The checkout must
contain the Meeting 2 package changes; publish/merge them before using the clone route.
The initial run downloads the pinned encoder and, if needed, about 90 MB of training CSVs.
No API key is needed. The official test files are never downloaded.

Original contributions and integration decisions: `docs/meeting2-integration.md`.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None
)
IN_COLAB = (
    importlib.util.find_spec("google.colab") is not None
    if importlib.util.find_spec("google")
    else False
)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/Poker-Decision-Model")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/nmokey/Poker-Decision-Model.git", str(ROOT)],
            check=True,
        )
    os.chdir(ROOT)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if not (ROOT / "src/poker_decision/training.py").is_file():
    raise RuntimeError("Use a checkout containing the Meeting 2 changes, then rerun.")
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[analysis,ml]"], check=True
    )
print("Project:", ROOT)

## 1. Choose a reproducible experiment

BERT Mini is the inexpensive default; switch `BACKBONE` to `minilm` or `distilbert` for
comparisons. Revisions are pinned. The total token limit includes state, question,
all actions, and separators. We **stop on overlength examples**, rather than truncate
history or silently drop decisions. Set a lower limit only after inspecting lengths.

The normal run reuses all saved Meeting 1 examples (normally 8,000 train / 2,000 validation).
For an end-to-end smoke run set `POKER_SMOKE=1` before launching Jupyter; this explicitly
samples 128 train / 64 validation examples and recomputes every baseline. Its scores
are not comparable with last week's published numbers. Each run uses a new output folder.

In [ ]:
import gc
import hashlib
import json
import platform
import random
from collections import Counter
from datetime import UTC, datetime
from importlib.metadata import version

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModel

from poker_decision.data import DecisionInputs
from poker_decision.evaluation import evaluate_by_street, inspect_mistakes, predicted_index
from poker_decision.models.encoder import (
    BACKBONES,
    QUESTION,
    PokerDecisionModel,
    encode_inputs,
    load_pretrained,
    model_input,
)
from poker_decision.pipeline import prepare, run_baselines, write_json, write_jsonl
from poker_decision.training import (
    batch_on_device,
    encode_examples,
    load_checkpoint,
    load_prepared,
    predict_encoded,
    prediction_lookup,
    save_checkpoint,
    score_model,
    seed_everything,
    train_epoch,
)

SEED = 42
BACKBONE = "bert_mini"
MODEL_NAME, MODEL_REVISION = BACKBONES[BACKBONE]
MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 3
TINY_N = 32
TINY_EPOCHS = 100
SMOKE = os.environ.get("POKER_SMOKE") == "1"
if SMOKE:
    EPOCHS = 1
seed_everything(SEED)
# Avoid oversubscribing CPU threads in local notebooks.
if not torch.cuda.is_available():
    torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / "artifacts/pokerbench_meeting2" / f"{BACKBONE}-{RUN_ID}"
OUT.mkdir(parents=True)
print("Device:", DEVICE, "| mode:", "smoke" if SMOKE else "working sample")
print("Backbone:", MODEL_NAME, MODEL_REVISION)
print("Outputs:", OUT)

## 2. Reuse the audited data and inspect an example

Load the saved Meeting 1 split; if absent, run its pinned, full-source validation and
preparation with the original defaults. Existing membership is checked, not resplit.
No future cards, source IDs, target indices, or reference answers enter the encoder.
`state` is the existing validated context string; converting it to nested JSON is optional.

In [ ]:
PROCESSED = ROOT / "data/processed/pokerbench_meeting1"
if not (PROCESSED / "manifest.json").exists():
    prepare(ROOT)
train, validation, manifest = load_prepared(ROOT)
if SMOKE:
    train = random.Random(SEED).sample(train, min(128, len(train)))
    validation = random.Random(SEED + 1).sample(validation, min(64, len(validation)))
assert {e.inputs.context for e in train}.isdisjoint(e.inputs.context for e in validation)
write_json(OUT / "data_manifest.json", manifest)
write_jsonl(OUT / "train.jsonl", [e.to_record() for e in train])
write_jsonl(OUT / "validation.jsonl", [e.to_record() for e in validation])
print("Examples:", len(train), "training;", len(validation), "validation")
display(
    pd.DataFrame(
        [
            {"split": name, "street": street, "n": n}
            for name, rows in [("train", train), ("validation", validation)]
            for street, n in sorted(Counter(e.street for e in rows).items())
        ]
    )
)
example = train[0]
print(json.dumps(model_input(example.inputs), indent=2))
print("Separate target:", example.target_index, example.target_action)

## 3. Tokenize one joint sequence and audit every example

Serialization: `[CLS] State … Question … Decisions: [SEP] option 1 [SEP] option 2 [SEP] …`.
Candidate spans include only the action's tokens. This requires no new special tokens.
Pretrained tokenizers may split cards and amounts into several pieces; inspect that
behavior rather than assuming each card is one token. Uncased backbones lowercase
text, so we retain the validated source spelling before tokenization and inspect its result.

In [ ]:
tokenizer, model = load_pretrained(MODEL_NAME, MODEL_REVISION)
MAX_LENGTH = min(MAX_LENGTH, int(model.encoder.config.max_position_embeddings))
for term in ["Ah", "Kd", "Ah Kd", "12bb", "3.5bb", "UTG", "raise 12bb"]:
    print(f"{term:12} -> {tokenizer.tokenize(term)}")
encoded = encode_inputs(example.inputs, tokenizer, max_length=MAX_LENGTH)
for action, (start, end) in zip(example.inputs.candidates, encoded.candidate_spans, strict=True):
    print(
        action,
        "positions",
        (start, end),
        "tokens",
        tokenizer.convert_ids_to_tokens(list(encoded.input_ids[start:end])),
    )
train_encoded = encode_examples(train, tokenizer, max_length=MAX_LENGTH)
val_encoded = encode_examples(validation, tokenizer, max_length=MAX_LENGTH)
lengths = pd.Series([len(e.input_ids) for e in train_encoded + val_encoded])
display(lengths.describe(percentiles=[0.5, 0.9, 0.99]).to_frame("tokens"))
print("All examples fit; none truncated or removed.")
PAD = tokenizer.pad_token_id

## 4. Inspect the model and a forward pass

The shared model is defined in `src/poker_decision/models/encoder.py`:

```python
hidden = encoder(input_ids, attention_mask).last_hidden_state  # [B, L, H]
pooled = candidate_token_mask @ hidden / candidate_token_counts  # [B, K, H]
logits = shared_linear(pooled).squeeze(-1)  # [B, K]
logits = logits.masked_fill(~valid_candidates, -inf)
```

There is one encoder sequence per poker example, containing every candidate. Candidate
slots and tokens are padded separately. Cross-entropy receives **raw logits**, not softmax.
The head is randomly initialized: these initial probabilities are not a trained strategy.

In [ ]:
model = model.to(DEVICE).eval()
batch = batch_on_device([encoded], PAD, DEVICE)
with torch.no_grad():
    logits = model(**batch)
    initial_probabilities = logits.softmax(-1)[0].cpu().tolist()
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Logit shape:", tuple(logits.shape))
display(
    pd.DataFrame(
        {"decision": example.inputs.candidates, "initial_probability": initial_probabilities}
    )
)
assert abs(sum(initial_probabilities) - 1) < 1e-6

## 5. Required learning check: overfit a tiny training set

Select 32 multi-option, distinct-context training examples in seeded order. This avoids
asking the debugging model to memorize contradictory labels for identical contexts.
Use a separate model, zero weight decay, and evaluate without dropout after each epoch.
This is a training-system check, not a claim of generalization. The full run starts fresh.
If it cannot reach 95% training accuracy, stop and investigate before scaling up.

In [ ]:
seen = set()
tiny = []
for e in random.Random(SEED).sample(train, len(train)):
    if len(e.inputs.candidates) > 1 and e.inputs.context not in seen:
        tiny.append(e)
        seen.add(e.inputs.context)
    if len(tiny) == TINY_N:
        break
if len(tiny) != TINY_N:
    raise ValueError("Need at least 32 distinct multi-option training contexts")
tiny_encoded = encode_examples(tiny, tokenizer, max_length=MAX_LENGTH)
# Release the untrained demonstration model before loading the tiny model.
del model
gc.collect()
seed_everything(SEED)
tiny_model = PokerDecisionModel(AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)).to(
    DEVICE
)
tiny_optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=1e-4, weight_decay=0)
tiny_history = []
for epoch in range(TINY_EPOCHS):
    loss = train_epoch(
        tiny_model,
        tiny_encoded,
        [e.target_index for e in tiny],
        pad_token_id=PAD,
        optimizer=tiny_optimizer,
        batch_size=8,
        seed=SEED + epoch,
    )
    metrics = score_model(tiny_model, tiny, tiny_encoded, pad_token_id=PAD)["overall"]
    tiny_history.append({"epoch": epoch + 1, "train_loss": loss, **metrics})
    if epoch == 0 or (epoch + 1) % 10 == 0 or metrics["exact_accuracy"] == 1:
        print(
            f"Tiny epoch {epoch + 1}: accuracy={metrics['exact_accuracy']:.1%}, "
            f"log loss={metrics['log_loss']:.4f}"
        )
    if metrics["exact_accuracy"] == 1:
        break
write_json(OUT / "tiny_history.json", tiny_history)
if tiny_history[-1]["exact_accuracy"] < 0.95:
    raise RuntimeError("Tiny learning check failed. Inspect inputs, labels, loss, and gradients.")
del tiny_model, tiny_optimizer
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 6. Train a fresh model; select the checkpoint by validation log loss

Fine-tune both encoder and head. Preserve source candidate order for this initial
experiment; the final section measures order sensitivity explicitly. Joint encoding
uses positional embeddings and does **not** guarantee permutation invariance.
We use ordinary cross-entropy before experimenting with class weighting, resampling,
or cross-validation. All baseline comparisons below use the exact same selected rows.

In [ ]:
seed_everything(SEED)
model = PokerDecisionModel(AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)).to(
    DEVICE
)
optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 2e-5},
        {"params": model.head.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.01,
)
metadata = {
    "backbone": MODEL_NAME,
    "revision": MODEL_REVISION,
    "question": QUESTION,
    "max_length": MAX_LENGTH,
    "serialization": "CLS + state/question + SEP + each candidate span/SEP; mean pooling",
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "encoder_lr": 2e-5,
    "head_lr": 1e-3,
    "weight_decay": 0.01,
    "mode": "smoke" if SMOKE else "working_sample",
    "device": str(DEVICE),
    "hardware": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.machine(),
    "python": platform.python_version(),
    "versions": {p: version(p) for p in ["torch", "transformers", "numpy", "scikit-learn"]},
    "dataset_revision": manifest["dataset_revision"],
    "official_test_used": False,
    "split_sha256": {
        name: hashlib.sha256((OUT / f"{name}.jsonl").read_bytes()).hexdigest()
        for name in ["train", "validation"]
    },
    "code_sha256": {
        str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted((ROOT / "src/poker_decision").rglob("*.py"))
    },
}
history = []
best_loss = float("inf")
for epoch in range(EPOCHS):
    train_loss = train_epoch(
        model,
        train_encoded,
        [e.target_index for e in train],
        pad_token_id=PAD,
        optimizer=optimizer,
        batch_size=BATCH_SIZE,
        seed=SEED + epoch,
    )
    metrics = score_model(model, validation, val_encoded, pad_token_id=PAD)["overall"]
    history.append({"epoch": epoch + 1, "train_loss": train_loss, **metrics})
    display(pd.DataFrame([history[-1]]))
    write_json(OUT / "training_history.json", history)
    if metrics["log_loss"] < best_loss:
        best_loss = metrics["log_loss"]
        save_checkpoint(
            OUT / "best",
            model,
            tokenizer,
            {**metadata, "best_epoch": epoch + 1, "validation_log_loss": best_loss},
        )
display(
    pd.DataFrame(history)
    .set_index("epoch")[["train_loss", "log_loss"]]
    .plot(ylabel="loss", title="Training and validation loss")
)

## 7. Reload the checkpoint and compare all four models

Checkpoint files contain the encoder, tokenizer, head, and experiment metadata.
Reload locally and check that predictions agree with the saved checkpoint before
using the standard Meeting 1 metrics. A single-option decision is forced; inspect
multi-option accuracy as well. Uniform exact accuracy uses lexical tie-breaking;
`tie_expected_accuracy` reports random choice among tied maxima.

Validation was used for checkpoint selection, so these are development scores.
Last week's 72.9% is only comparable when membership is identical; the table below
recomputes baselines rather than copying old scores. Official-test evaluation is deferred.

In [ ]:
del model, optimizer
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
reloaded_tokenizer, model, checkpoint_metadata = load_checkpoint(OUT / "best")
model = model.to(DEVICE)
reloaded_encoded = encode_examples(validation, reloaded_tokenizer, max_length=MAX_LENGTH)
assert reloaded_encoded == val_encoded
model_probs = predict_encoded(model, val_encoded, pad_token_id=PAD)
# A second independent local reload verifies both encoder and head persistence.
_, check_model, _ = load_checkpoint(OUT / "best")
check_model = check_model.to(DEVICE)
check_probs = predict_encoded(check_model, val_encoded[:4], pad_token_id=PAD, batch_size=4)
for expected, actual in zip(model_probs[:4], check_probs, strict=True):
    torch.testing.assert_close(torch.tensor(expected), torch.tensor(actual), atol=1e-5, rtol=1e-4)
del check_model
gc.collect()
lookup = prediction_lookup(validation, model_probs)
report = run_baselines(OUT, train, validation, seed=SEED)
comparison = dict(report["metrics"])
comparison["transformer"] = evaluate_by_street(validation, lookup.__getitem__)
write_json(OUT / "comparison.json", comparison)
display(
    pd.DataFrame(
        [
            {"model": name, "street": street, **metrics}
            for name, groups in comparison.items()
            for street, metrics in groups.items()
        ]
    ).set_index(["model", "street"])
)
write_jsonl(
    OUT / "predictions.jsonl",
    [{**e.to_record(), "probabilities": p} for e, p in zip(validation, model_probs, strict=True)],
)

## 8. Inspect five mistakes and probe candidate order

Reverse candidates on up to 64 validation examples, rerun the full joint encoder,
and align probabilities back by action. Report probability changes and prediction
flips; do not assume exact invariance. Fixed-position shortcuts are a reason to try
candidate shuffling during training in the next experiment.

In [ ]:
mistakes = inspect_mistakes(validation, lookup.__getitem__, seed=SEED)
write_jsonl(OUT / "five_errors.jsonl", mistakes)
for mistake in mistakes:
    print(mistake["context"])
    print("Reference:", mistake["target_action"], "| predicted:", mistake["predicted_action"])
    display(
        pd.DataFrame({"decision": mistake["candidates"], "probability": mistake["probabilities"]})
    )
probe = validation[: min(64, len(validation))]
reversed_inputs = [
    DecisionInputs(e.inputs.context, tuple(reversed(e.inputs.candidates))) for e in probe
]
reversed_encoded = [encode_inputs(x, tokenizer, max_length=MAX_LENGTH) for x in reversed_inputs]
reversed_probs = predict_encoded(model, reversed_encoded, pad_token_id=PAD)
deltas, flips = [], []
for e, x, p in zip(probe, reversed_inputs, reversed_probs, strict=True):
    original = lookup[e.inputs]
    aligned = list(reversed(p))
    deltas.append(max(abs(a - b) for a, b in zip(original, aligned, strict=True)))
    flips.append(
        e.inputs.candidates[predicted_index(e.inputs, original)]
        != x.candidates[predicted_index(x, p)]
    )
order_report = {
    "n": len(probe),
    "mean_max_probability_change": sum(deltas) / len(deltas),
    "prediction_flip_rate": sum(flips) / len(flips),
}
write_json(OUT / "candidate_order.json", order_report)
print(order_report)
print(
    "Share the notebook, encoder/revision, comparison.json, "
    "training history, and five_errors.jsonl."
)
print("Checkpoint and all generated results:", OUT)

## What have we demonstrated?

- A tiny training-set overfit checks that the learning system works.
- Held-out agreement and probability metrics test transfer to other examples.
- Menu-only and linear baselines help assess whether the transformer adds value.
- Per-street metrics, errors, and candidate-order probes identify the next experiment.

Exact-context separation does not isolate suit-equivalent hands or related scenarios.
Source menus may contain artifacts. Matching the reference action is not evidence of
profitable or unexploitable play, and softmax outputs are not solver mixing frequencies.
Do not add arbitrary tilt/opponent descriptions to unchanged labels and claim adaptation.

Next experiments: candidate-order augmentation, stricter scenario grouping, tokenizer/card
representations, and additional backbone comparisons. Keep one maintained walkthrough;
put reusable improvements in the package with offline tests.